In [21]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder

import pandas as pd
from collections import Counter

BATCH_SIZE = 32

In [2]:
url = "https://raw.githubusercontent.com/spyguessgame-boop/own_dataset/refs/heads/main/emotion_dataset.csv"

df = pd.read_csv(url)

encoder = LabelEncoder()
df['Output'] = encoder.fit_transform(df['emotion'])

df.head()

,text,emotion,Output
0,I always knew that I talked briefly with my te...,Normal,2
1,"Honestly, I reviewed some documents and provid...",Normal,2
2,"At this point, I feel hopeless and don't know ...",Sad,3
3,"I have to say, Life feels so beautiful and ful...",Happy,1
4,"Honestly, I threw the papers across the room w...",Anger,0


In [3]:
classes = encoder.classes_
classes

array(['Anger', 'Happy', 'Normal', 'Sad'], dtype=object)

## Build Vocabulary

In [4]:
def build_vocabulary(texts,min_freq=2) :
    counter = Counter()
    for text in texts :
        counter.update(word.lower() for word in text.split())
    vocab = {"<PAD>": 0, "<UNK>": 1}


    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocabulary(df["text"])
print("Vocab Size" , len(vocab))


Vocab Size 692


## Text to Number

In [5]:
def encode(text, vocab, max_len=20):
    tokens = text.split()

    encoded = [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

    # padding
    if len(encoded) < max_len:
        encoded += [vocab["<PAD>"]] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]

    return encoded



## Create PyTorch Dataset

In [6]:
class EmotionDataset(Dataset):
    def __init__(self, df , vocab) :
        self.texts = df["text"].values
        self.labels = df["Output"].values
        self.vocab = vocab

    def __len__(self) :
        return len(self.texts)
    def __getitem__(self, idx) :
        text = self.texts[idx]
        label = self.labels[idx]

        x = encode(text, self.vocab)

        return torch.tensor(x, dtype=torch.long), torch.tensor(label)

## DataLoader

In [7]:
def encode(text, vocab, max_len=20):
    tokens = text.split()

    encoded = [
        vocab.get(token.lower(), vocab["<UNK>"])
        for token in tokens
    ]

    # padding
    if len(encoded) < max_len:
        encoded += [vocab["<PAD>"]] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]

    return encoded

dataset = EmotionDataset(df,vocab)

dataloader = DataLoader(
    dataset ,
    batch_size = BATCH_SIZE,
    shuffle = True
)

for x,y in dataloader :
    print(x.shape) ## (batch_size, seq_len)
    print(y.shape) # (Batch_size)
    break

torch.Size([32, 20])
torch.Size([32])


## Embeding + Linear Layer (Simple First)

In [8]:
class SimpleEmbeddingModel(nn.Module):
    def __init__(self, vocab_size , embed_dim , num_classes) :
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, num_classes)


    def forward(self, x) :
        x = self.emb(x)
        x = x.mean(dim = 1)

        return self.linear(x)

vocab_size = len(vocab)
embed_dim  = 64

model = SimpleEmbeddingModel(vocab_size,64 , 4)
model

SimpleEmbeddingModel(
  (emb): Embedding(692, 64)
  (linear): Linear(in_features=64, out_features=4, bias=True)
)

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    for x, y in dataloader:

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 1.2033478021621704
Epoch 2, Loss: 0.7715682983398438
Epoch 3, Loss: 0.4025661051273346
Epoch 4, Loss: 0.17624308168888092
Epoch 5, Loss: 0.07475539296865463
Epoch 6, Loss: 0.06407753378152847
Epoch 7, Loss: 0.03500537946820259
Epoch 8, Loss: 0.027732491493225098
Epoch 9, Loss: 0.02030596137046814
Epoch 10, Loss: 0.01804989203810692


In [10]:
input_text = "I am happy"
encoded_text = encode(input_text, vocab)
input_tensor = torch.tensor(encoded_text, dtype=torch.long).unsqueeze(0)

model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculations
    output = model(input_tensor)

predicted_class_idx = torch.argmax(output).item()
predicted_emotion = classes[predicted_class_idx]
print(f"The predicted emotion for '{input_text}' is: {predicted_emotion}")

The predicted emotion for 'I am happy' is: Anger


## LSTM Model

In [11]:
class LSTMModel(nn.Module) :
    def __init__(self , vocab_size , emb_dim , hidden_dim , num_classes) :
        super().__init__()

        self.emb = nn.Embedding(vocab_size, emb_dim)

        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

        self.linear = nn.Linear(hidden_dim, num_classes)

    def forward(self,x) :
        x = self.emb(x)

        lstm_out, _ = self.lstm(x)

        # Take mean of all time steps
        mean_out = lstm_out.mean(dim=1)

        output = self.linear(mean_out)
        return output

hidden_dim = 128

model0 = LSTMModel(
    vocab_size=len(vocab),
    emb_dim=64,
    hidden_dim=hidden_dim,
    num_classes=len(set(df["Output"]))
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model0.parameters(), lr=0.001)

for epoch in range(10):
    for x, y in dataloader:

        optimizer.zero_grad()

        outputs = model0(x)

        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.09886982291936874
Epoch 2, Loss: 0.0052716596983373165
Epoch 3, Loss: 0.004677803721278906
Epoch 4, Loss: 0.0008078893297351897
Epoch 5, Loss: 0.000921755563467741
Epoch 6, Loss: 0.0005844656261615455
Epoch 7, Loss: 0.0010381604079157114
Epoch 8, Loss: 0.00023164943559095263
Epoch 9, Loss: 0.00017985061276704073
Epoch 10, Loss: 0.0002848220756277442


In [12]:


input_text = "Today was a beautiful day and I felt very happy inside"
input_text = input_text.lower()
encoded_text = encode(input_text, vocab)
input_tensor = torch.tensor(encoded_text, dtype=torch.long).unsqueeze(0)

model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculations
    output = model0(input_tensor)
    print(output)

predicted_class_idx = torch.argmax(output).item()
predicted_emotion = classes[predicted_class_idx]
print(f"The predicted emotion for '{input_text}' is: {predicted_emotion}")

tensor([[-4.6253, 10.8158, -3.5792, -3.6210]])
The predicted emotion for 'today was a beautiful day and i felt very happy inside' is: Happy


In [13]:
def predict(text):
    text = text.lower()  # MUST

    encoded = encode(text, vocab)
    x = torch.tensor(encoded).unsqueeze(0)

    output = model(x)
    pred = torch.argmax(output, dim=1)

    return pred.item()
print(encode("i am happy", vocab))

[2, 544, 235, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [14]:
sample = df.iloc[0]["text"]
label = df.iloc[0]["Output"]

print(sample)
print("Actual:", label)
print("Pred:", predict(sample))

I always knew that I talked briefly with my teacher about the upcoming plans.
Actual: 2
Pred: 2


In [15]:
print(vocab.get("happy"))
print(vocab.get("sad"))
print(vocab.get("angry"))

235
435
197


# Attention layer

In [16]:
class SelfAttention(nn.Module) :
    def __init__(self, embed_dim) :
        super().__init__()

        self.query = nn.Linear(embed_dim , embed_dim)
        self.key = nn.Linear(embed_dim , embed_dim)
        self.value = nn.Linear(embed_dim , embed_dim)

    def forward(self ,x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Attention scores
        scores = torch.matmul(Q , K.transpose(-2,-1))

        # Scale (VERY IMPORTANT 🔥)
        scores = scores / (x.shape[-1] ** 0.5)

        weights = F.softmax(scores , dim=-1)

        out = torch.matmul(weights , V)

        return out

In [37]:
class TransformerMini(nn.Module) :
    def __init__(self, vocab_size , emb_dim , num_classes) :
        super().__init__()

        self.emb = nn.Embedding(vocab_size , emb_dim)

        self.attention = SelfAttention(emb_dim)

        self.linear = nn.Linear(emb_dim , num_classes)

    def forward(self,x) :
        embedded = self.emb(x)

        att_out = self.attention(embedded)

        mean_out = att_out.mean(dim=1)

        output = self.linear(mean_out)

        return output

model1 = TransformerMini(
    vocab_size=len(vocab),
    emb_dim=64,
    num_classes=len(set(df["Output"]))
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model1.parameters(), lr=0.01)

In [38]:
for i in range(10) :
    for x,y in dataloader :
        optimizer.zero_grad()

        output = model1(x)

        loss = criterion(output , y)

        loss.backward()

        optimizer.step()

    print(f"For {i} loss===> {loss.item()}")

For 0 loss===> 8.629961666883901e-05
For 1 loss===> 7.76683518779464e-05
For 2 loss===> 7.385899607470492e-06
For 3 loss===> 8.816364243102726e-06
For 4 loss===> 4.84284237245447e-06
For 5 loss===> 1.2744200830638874e-05
For 6 loss===> 6.581210982403718e-06
For 7 loss===> 2.7418016088631703e-06
For 8 loss===> 1.1507888302730862e-05
For 9 loss===> 3.993460268247873e-06


In [40]:
# input_text = "Today was a beautiful day and I felt very happy inside"
input_text = "I am sad"
input_text = input_text.lower()
encoded_text = encode(input_text, vocab)
input_tensor = torch.tensor(encoded_text, dtype=torch.long).unsqueeze(0)

model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculations
    output = model1(input_tensor)
    print(output)

predicted_class_idx = torch.argmax(output).item()
predicted_emotion = classes[predicted_class_idx]
print(f"The predicted emotion for '{input_text}' is: {predicted_emotion}")

tensor([[ -5.4425,   5.6801, -11.2918,   9.9476]])
The predicted emotion for 'i am sad' is: Sad
